In [ ]:
import pandas as pd

In [ ]:
xl = pd.ExcelFile("../Pomiary Z Ciepłomierzy/Aktualne v2/APEK.xlsx")

In [ ]:
xl.sheet_names

In [ ]:
df = xl.parse("S202504")

In [ ]:
df

In [ ]:
df["date"] = df["pollub/MB"]

In [ ]:
df["time"] = df["Unnamed: 1"]

In [ ]:
df

In [ ]:
df = df.iloc[556:]

In [ ]:
from datetime import datetime

In [ ]:
df["time2"] = df["time"].map(lambda x : x.time() if isinstance(x, datetime) else x  )

In [ ]:
df["original_time"] = df["Unnamed: 2"].map(lambda x : x.time() if isinstance(x, datetime) else x  )

In [ ]:

# Condition: time2 occurs earlier in the day than original_time
mask = df['time2'] < df['original_time']

df["date_fixed"] = df["date"]

# Add 1 day where condition is met
df.loc[mask, 'date_fixed'] = df.loc[mask, 'date'] + pd.Timedelta(days=1)

In [ ]:
df["ts"] = pd.to_datetime(df["date_fixed"].astype("str") + " " + df["time2"].astype("str"))

In [ ]:
df

In [ ]:
import pandas as pd

# Ensure ts is datetime
df['ts'] = pd.to_datetime(df['ts'])

# Identify rows where ts is not increasing
problem_idx = df.index[df['ts'] <= df['ts'].shift(1)]

# Include the previous row for context
context_idx = problem_idx.union(problem_idx - 1)

# Display a handful of cases, ordered
print(df.loc[context_idx].sort_index().head(10))


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # try this first


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure ts is datetime
df['ts'] = pd.to_datetime(df['ts'])

# Create the plot
plt.figure(figsize=(50, 8))
plt.plot(df['ts'], df['T1'])
plt.xlabel('Timestamp')
plt.ylabel('T1')
plt.title('T1 over Time')

# Improve x-axis formatting
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


In [ ]:
import pandas as pd
import plotly.express as px

df['ts'] = pd.to_datetime(df['ts'])

fig = px.line(df, x='ts', y='T1', title='T1 over Time')

fig.update_xaxes(
    rangeslider_visible=True
)

fig.show()

In [ ]:
mask = (df["ts"] > "2025-04-15") & (df["ts"] < "2025-04-20") 

In [ ]:
df[mask]

In [ ]:
import pandas as pd
import plotly.express as px

df["ts"] = pd.to_datetime(df["ts"])

df["czas"] = df["ts"]

fig = px.line(df, x="czas", y="T1", title="Temperatura przed zbiornikiem")
fig.update_xaxes(rangeslider_visible=True)

fig.write_html("plot.html", include_plotlyjs="cdn")  # smaller file; loads plotly.js from CDN


In [ ]:
import pandas as pd
import plotly.express as px

# Ensure datetime
df["ts"] = pd.to_datetime(df["ts"])
df["czas"] = df["ts"]

# Reshape so T1 and T2 are plotted together
df_long = df.melt(
    id_vars="czas",
    value_vars=["T1", "T2", "T3", "T4", "Tzew", "Twew", "T2-T1"], # no ua
    var_name="Sensor",
    value_name="Temperature"
)

# Plot
fig = px.line(
    df_long,
    x="czas",
    y="Temperature",
    color="Sensor",
    title="Temperatura przed zbiornikiem",
    labels={
        "czas": "Czas",
        "Temperature": "T [°C]"
    }
)

fig.update_xaxes(rangeslider_visible=True)

# Save as interactive HTML
fig.write_html("plot.html", include_plotlyjs="cdn")


In [ ]:
xl_belimo = pd.ExcelFile("../Pomiary Z Ciepłomierzy/Aktualne v2/Belimo co 1 min.xlsx")

In [ ]:
xl_belimo.sheet_names

In [ ]:
df_belimo = xl_belimo.parse("Belimo")

In [ ]:
df_belimo.columns

In [ ]:
df_belimo["czas"] = df_belimo["Data"].astype("str") + " " + df_belimo["Godzina"].astype("str")

In [ ]:
df_belimo[["czas", "Data", "Energia T1, GJ"]].groupby("Data")

In [ ]:
df_dobowe_zuzycie_ciepla = df_belimo.loc[df_belimo.groupby("Data")["czas"].idxmin()]

In [ ]:
df_belimo["czas"]

In [ ]:
xl_kamstrup6 = pd.ExcelFile("../Pomiary Z Ciepłomierzy/Aktualne v2/Kamstrup 206.xlsx")

In [ ]:
xl_kamstrup6.sheet_names

In [ ]:
df_kamstrup6 = xl_kamstrup6.parse("Sheet1") 

In [ ]:
df_kamstrup6

In [ ]:
df_kamstrup6["czas"] =  df_kamstrup6["Numer klienta"]

In [ ]:
df_kamstrup6 = df_kamstrup6.iloc[2:]

In [ ]:
df_kamstrup6["czas"]

In [ ]:
df_kamstrup6 = df_kamstrup6[(df_kamstrup6["czas"] > pd.Timestamp("2025-04-05 0:00")) & (df_kamstrup6["czas"] < pd.Timestamp("2025-07-04 23:59"))]

In [ ]:
xl_kamstrup7 = pd.ExcelFile("../Pomiary Z Ciepłomierzy/Aktualne v2/Kamstrup 207.xlsx")

In [ ]:
xl_kamstrup7.sheet_names

In [ ]:
df_kamstrup7 = xl_kamstrup7.parse('Sheet1 (2)')

In [ ]:
df_kamstrup7["czas"] =  df_kamstrup7["Numer klienta"]

In [ ]:
df_kamstrup7  = df_kamstrup7.iloc[2:]

In [ ]:
df_kamstrup7

In [ ]:
df_kamstrup7 = df_kamstrup7[(df_kamstrup7["czas"] > pd.Timestamp("2025-04-05 0:00")) & (df_kamstrup7["czas"] < pd.Timestamp("2025-07-04 23:59"))]

In [ ]:
xl_pec = pd.ExcelFile("../Pomiary Z Ciepłomierzy/Aktualne v2/Ciepłomierz główny PECu.xlsx")

In [ ]:
xl_pec.sheet_names

In [ ]:
df_pec = xl_pec.parse('Ciepłomierz główny PECu')

In [ ]:
df_pec["czas"] = df_pec["Data odczytu"]

In [ ]:
df_pec["T1"] = df_pec["Temperatura zasilania [°C]"].str.replace(",", ".", regex=False)

In [ ]:
df_pec["T2"] = df_pec["Temperatura powrotu [°C]"].str.replace(",", ".", regex=False)

In [ ]:
df_pec["T1-T2"] = df_pec["Różnica temperatur [°C]"].str.replace(",", ".", regex=False)

In [ ]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

# ---------- Prepare data ----------
df["ts"] = pd.to_datetime(df["ts"])
df["czas"] = df["ts"]
df_belimo["czas"] = pd.to_datetime(df_belimo["czas"])

df_long = df.melt(
    id_vars="czas",
    value_vars=["T1", "T2", "T3", "T4", "Tzew", "Twew", "T2-T1"],
    var_name="Sensor",
    value_name="Temperature",
)

df_belimo_temp = df_belimo.melt(
    id_vars="czas",
    value_vars=["T1, st.C", "T2, st.C"],
    var_name="Sensor",
    value_name="Temperature",
)

df_belimo_energia = df_belimo.melt(
    id_vars="czas",
    value_vars=["Energia T1, GJ", "Energia T2, GJ"],
    var_name="Sensor",
    value_name="Energia",
)

df_pec_temp = df_pec.melt(
    id_vars="czas",
    value_vars=["T1", "T2", "T1-T2"],
    var_name="Sensor",
    value_name="Temperature",
)

plots = [
    dict(
        title="APEK - Temperatura [°C]",
        fig=px.line(df_long, x="czas", y="Temperature", color="Sensor"),
        y_title="°C",
        showlegend=True,   # ✅ keep legend
        legendgroup="APEK",  # helps organization
    ),
    dict(
        title="Kamstrup - Energia Cieplna [GJ]",
        fig=None,                # we'll add traces manually
        y_title="GJ",
        showlegend=True,
    ),

    dict(
        title="Kamstrup - Objętość [m³]",
        fig=None,                # we'll add traces manually
        y_title="m³",
        showlegend=True,
    ),

    dict(
        title="Kamstrup - Temperatura [°C]",
        fig=None,                # we'll add traces manually
        y_title="°C",
        showlegend=True,
    ),
    dict(
        title="Belimo — Przepływ [m³/h]",
        fig=px.line(df_belimo, x="czas", y="Przepływ, m3/h"),
        y_title="m³/h",
        showlegend=False,  # hide legend
    ),
    dict(
        title="Belimo — Przepływ [%]",
        fig=px.line(df_belimo, x="czas", y="Przepływ, %"),
        y_title="%",
        showlegend=False,  # hide legend
    ),
    dict(
        title="Belimo — Temperatura [°C]",
        fig=px.line(df_belimo_temp, x="czas", y="Temperature", color="Sensor"),
        y_title="°C",
        showlegend=True,    # ✅ keep legend
        legendgroup="Belimo",
    ),
    dict(
        title="Belimo — Moc [W]",
        fig=px.line(df_belimo, x="czas", y="Moc, W"),
        y_title="W",
        showlegend=False,  # hide legend
    ),
    dict(
        title="Belimo — Delta T [K]",
        fig=px.line(df_belimo, x="czas", y='DeltaT, K'),
        y_title="K",
        showlegend=False,  # hide legend
    ),
    dict(
        title="Belimo — Energia [GJ]",
        fig=px.line(df_belimo_energia, x="czas", y="Energia", color="Sensor"),
        y_title="GJ",
        showlegend=True,    # ✅ keep legend
        legendgroup="BelimoEnergia",
    ),
    
    dict(
        title="Belimo — Objętość V [m³]",
        fig=px.line(df_belimo, x="czas", y='Objętość V, m3'),
        y_title="m³",
        showlegend=False,  # hide legend
    ), 
    
    dict(
        title="PEC — Objętość V [m³]",
        fig=px.line(df_pec, x="czas", y='Objętość [m3]'),
        y_title="m³",
        showlegend=False,  # hide legend
    ),

    dict(
        title="PEC — Przepływ [dm³/h]",
        fig=px.line(df_pec, x="czas", y='Przepływ [l/h]'),
        y_title="dm³/h",
        showlegend=False,  # hide legend
    ),

    dict(
        title="PEC — Moc [kW]",
        fig=px.line(df_pec, x="czas", y='Moc [kW]'),
        y_title="kW",
        showlegend=False,  # hide legend
    ),
    dict(
        title="PEC - Temperatura [°C]",
        fig=px.line(df_pec_temp, x="czas", y="Temperature", color="Sensor"),
        y_title="°C",
        showlegend=True,   # ✅ keep legend
        legendgroup="PEC",  # helps organization
    ),
]

belimo_colors = {
    "T1, st.C": "#d62728",
    "T2, st.C": "#1f77b4",
}





rows = len(plots) 

fig = make_subplots(
    rows=rows,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
    row_heights=[0.88 / len(plots)] * len(plots),
    subplot_titles=[p["title"] for p in plots],
)

for i, p in enumerate(plots, start=1):
    
    # 🔹 special case: Kamstrup subplot (last row)
    if p["title"] == "Kamstrup - Energia Cieplna [GJ]":
        fig.add_scatter(
            x=df_kamstrup6["czas"],
            y=df_kamstrup6[85349206],
            mode="lines",
            name="Kamstrup 206",
            legendgroup=p["title"],
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )
    
        fig.add_scatter(
            x=df_kamstrup7["czas"],
            y=df_kamstrup7[85349207],
            mode="lines",
            name="Kamstrup 207",
            legendgroup=p["title"],
            # title only needs to be on one trace, but harmless if repeated
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )
    
        fig.update_yaxes(title_text=p["y_title"], row=i, col=1)

        continue  # skip normal px-figure handling

     # 🔹 special case: Kamstrup subplot (last row)
    if p["title"] == "Kamstrup - Objętość [m³]":
        fig.add_scatter(
            x=df_kamstrup6["czas"],
            y=df_kamstrup6["Unnamed: 8"],
            mode="lines",
            name="Kamstrup 206",
            legendgroup=p["title"],
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )
    
        fig.add_scatter(
            x=df_kamstrup7["czas"],
            y=df_kamstrup7["Unnamed: 8"],
            mode="lines",
            name="Kamstrup 207",
            legendgroup=p["title"],
            # title only needs to be on one trace, but harmless if repeated
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )
    
        fig.update_yaxes(title_text=p["y_title"], row=i, col=1)

        continue  # skip normal px-figure handling

     # 🔹 special case: Kamstrup subplot (last row)
    if p["title"] == "Kamstrup - Temperatura [°C]":
        fig.add_scatter(
            x=df_kamstrup6["czas"],
            y=df_kamstrup6["Unnamed: 18"],
            mode="lines",
            name="Kamstrup 206 - T1",
            legendgroup=p["title"],
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )

        fig.add_scatter(
            x=df_kamstrup6["czas"],
            y=df_kamstrup6["Unnamed: 19"],
            mode="lines",
            name="Kamstrup 206 - T2",
            legendgroup=p["title"],
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )
    
        fig.add_scatter(
            x=df_kamstrup7["czas"],
            y=df_kamstrup7["Unnamed: 18"],
            mode="lines",
            name="Kamstrup 207 - T1",
            legendgroup=p["title"],
            # title only needs to be on one trace, but harmless if repeated
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )

        fig.add_scatter(
            x=df_kamstrup7["czas"],
            y=df_kamstrup7["Unnamed: 19"],
            mode="lines",
            name="Kamstrup 207 - T2",
            legendgroup=p["title"],
            # title only needs to be on one trace, but harmless if repeated
            legendgrouptitle=dict(text=p["title"]),
            row=i, col=1,
        )
    
        fig.update_yaxes(title_text=p["y_title"], row=i, col=1)

        continue  # skip normal px-figure handling
    
    for tr in p["fig"].data:
        tr.showlegend = bool(p.get("showlegend", True))

        # ---- Belimo temperature subplot ----
        if  p["title"] == "Belimo — Temperatura [°C]":
            original = tr.name  # "T1, st.C" / "T2, st.C"

            # keep unique, clickable legend items
            tr.name = original.replace(", st.C", "")

            # group legend with a header
            tr.legendgroup = "BelimoTemp"
            tr.legendgrouptitle = dict(text= p["title"])

            # fixed colors
            tr.line.color = belimo_colors.get(original)

            
        # ---- Belimo energia subplot ----
        if p["title"]  == "Belimo — Energia [GJ]":
            original = tr.name  # "T1, st.C" / "T2, st.C"

            # keep unique, clickable legend items
            tr.name = original.replace(", st.C", "")

            # group legend with a header
            tr.legendgroup = "BelimoEnergia"
            tr.legendgrouptitle = dict(text= p["title"])

            # fixed colors
            tr.line.color = belimo_colors.get(original)

        if p["title"] == "APEK - Temperatura [°C]":
            original = tr.name  # "T1, st.C" / "T2, st.C"

            # keep unique, clickable legend items
            tr.name = original.replace(", st.C", "")

            # group legend with a header
            tr.legendgroup = "APEK"
            tr.legendgrouptitle = dict(text= p["title"])

        
        if p["title"] == "PEC - Temperatura [°C]":
            original = tr.name  # "T1, st.C" / "T2, st.C"

            # keep unique, clickable legend items
            tr.name = original.replace(", st.C", "")

            # group legend with a header
            tr.legendgroup = "PEC"
            tr.legendgrouptitle = dict(text= p["title"])

        fig.add_trace(tr, row=i, col=1)

    fig.update_yaxes(
        title_text=p.get("y_title", ""),
        showticklabels=True,
        automargin=True,
        row=i, col=1
    )
        
fig.update_layout(
    title="",
    height=300 * rows,  # 🔼 bigger charts
    hovermode="x unified",
    legend_title_text="",
    legend=dict(groupclick="toggleitem"),
    margin=dict(l=70, r=15, t=25, b=35),  # 🔽 tighter margins
)


# Apply slider ONLY to the bottom shared x-axis
#fig.update_layout(xaxis=dict(rangeslider=dict(visible=True, thickness=0.035)))

# Give y-axes room (so labels don't get clipped)
fig.update_layout(margin=dict(l=70, r=20, t=60, b=50))

fig.update_xaxes(
    showticklabels=True,
    ticks="outside",
    tickfont=dict(size=10),
)

fig.update_xaxes(
    rangeslider=dict(
        visible=True,
        thickness=0.01,
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
    ),
    row=rows,
    col=1,
)


fig.update_yaxes(autorange=True)
fig.update_yaxes(type="linear")

fig.update_layout(margin=dict(l=70, r=20, t=60, b=50))

fig.write_html("plot.html", include_plotlyjs="cdn")


In [ ]:
# Wstrzykuje jasny, „bootstrapowy” panel instrukcji do istniejącego pliku HTML (np. plot.html)
# Nie zmienia wykresu — dodaje tylko sekcję instrukcji na górze strony.

from pathlib import Path

PLOT_HTML_PATH = "plot.html"                 # wejściowy plik HTML
OUTPUT_HTML_PATH = "plot_z_instrukcja.html"  # wynik

INSTRUKCJA_HTML = r"""
<style>
  /* Jasne tło strony + lekko bootstrapowy wygląd */
  body{
  background:#f8f9fa !important;
  color:#212529 !important;
  font-family:
    system-ui,
    -apple-system,
    "Segoe UI",
    Roboto,
    "Helvetica Neue",
    Arial,
    "Noto Sans",
    "Liberation Sans",
    sans-serif !important;
}
  .container-ish{
    max-width: 1080px;
    margin: 16px auto 10px;
    padding: 0 16px;
  }
  .cardish{
    background:#ffffff;
    border:1px solid rgba(0,0,0,.125);
    border-radius:.5rem;
    box-shadow: 0 .125rem .25rem rgba(0,0,0,.075);
    overflow:hidden;
  }
  .cardish-header{
    padding: .75rem 1rem;
    background:#ffffff;
    border-bottom:1px solid rgba(0,0,0,.125);
    display:flex;
    align-items:center;
    justify-content:space-between;
    gap: 10px;
  }
  .cardish-title{
    margin:0;
    font-size: 1rem;
    font-weight: 600;
  }
  .badgeish{
    display:inline-block;
    padding:.25rem .5rem;
    border-radius: 999px;
    font-size: .75rem;
    border: 1px solid rgba(0,0,0,.125);
    background:#f1f3f5;
    color:#495057;
    white-space: nowrap;
  }
  .cardish-body{
    padding: 1rem;
  }
  .muted{
    color:#6c757d;
    margin: 0 0 .75rem 0;
    line-height: 1.45;
  }
  .rowish{
    display:grid;
    grid-template-columns: 1fr;
    gap: .75rem;
  }
  @media (min-width: 900px){
    .rowish{ grid-template-columns: 1fr 1fr; }
  }
  .listish{
    margin: 0;
    padding-left: 1.1rem;
  }
  .listish li{
    margin: .35rem 0;
    line-height: 1.45;
  }
  .kbdish{
    display:inline-block;
    padding: .1rem .4rem;
    border-radius: .35rem;
    border:1px solid rgba(0,0,0,.2);
    background:#f8f9fa;
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono","Courier New", monospace;
    font-size: .85em;
    white-space: nowrap;
  }
  .alertish{
    margin-top: .75rem;
    padding: .75rem 1rem;
    border-radius: .5rem;
    border: 1px solid rgba(13,110,253,.25);
    background: rgba(13,110,253,.08);
    color:#084298;
    line-height:1.45;
  }

  /* Akordeon (details/summary) w stylu bootstrap */
  details.accordionish > summary{
    list-style:none;
    cursor:pointer;
    user-select:none;
  }
  details.accordionish > summary::-webkit-details-marker{ display:none; }
  .chev{
    font-size: 1rem;
    color:#6c757d;
    transition: transform .15s ease-in-out;
  }
  details[open] .chev{ transform: rotate(180deg); }

  /* Trochę miejsca nad wykresem (żeby panel nie "przyklejał się") */
  .container-ish + div, .container-ish + main { margin-top: 6px; }
</style>

<div class="container-ish" id="instrukcja-obslugi">
  <details class="accordionish cardish" open>
    <summary class="cardish-header">
      <div style="display:flex;align-items:center;gap:.5rem;flex-wrap:wrap;">
        <h2 class="cardish-title">Instrukcja obsługi wykresu</h2>
        <span class="badgeish">kliknij, aby zwinąć/rozwinąć</span>
      </div>
      <span class="chev">⌄</span>
    </summary>

    <div class="cardish-body">
      <p class="muted">
        Wykres jest interaktywny. Poniżej znajdziesz krótką ściągę: jak przybliżać, przesuwać widok i sterować seriami z legendy.
      </p>

      <div class="rowish">
        <div>
          <strong>Na samym wykresie</strong>
          <ul class="listish">
            <li><b>Podgląd wartości:</b> przesuń kursor po wykresie.</li>
            <li><b>Przybliżenie (zoom):</b> kliknij i przeciągnij myszą po obszarze do powiększenia.</li>
            <li><b>Przesuwanie widoku:</b> złap wykres i przeciągnij, aby przesunąć w czasie.</li>
            <li><b>Reset widoku:</b> <span class="kbdish">podwójny klik</span> w obszar wykresu.</li>
            <li><b>Kółko myszy:</b> w wielu przypadkach przybliża/oddala (jeśli nie — użyj zoomu prostokątem).</li>
          </ul>
        </div>

        <div>
          <strong>Legenda i suwak czasu</strong>
          <ul class="listish">
            <li><b>Ukryj/pokaż serię:</b> kliknij jej nazwę w legendzie.</li>
            <li><b>Pokaż tylko jedną serię:</b> <span class="kbdish">dwuklik</span> na nazwie w legendzie.</li>
            <li><b>Przywróć wszystkie serie:</b> ponownie <span class="kbdish">dwuklik</span> (albo reset widoku).</li>
            <li><b>Suwak czasu:</b> przeciągnij lewy/prawy uchwyt, aby ustawić zakres; przeciągnij zaznaczenie, aby przesunąć zakres.</li>
          </ul>
        </div>
      </div>

      <div class="alertish">
        Wskazówka: jeśli przeciąganie „zamiast przesuwać” robi przybliżenie (lub odwrotnie),
        skorzystaj z ikon w prawym górnym rogu wykresu (tryb przesuwania / tryb przybliżania).
      </div>
    </div>
  </details>
</div>
"""

def inject_panel_into_html(input_path: str, output_path: str, injection_html: str) -> None:
    html = Path(input_path).read_text(encoding="utf-8")

    # zabezpieczenie przed podwójnym wstrzyknięciem
    if 'id="instrukcja-obslugi"' in html:
        Path(output_path).write_text(html, encoding="utf-8")
        return

    lower = html.lower()
    body_idx = lower.find("<body")
    if body_idx == -1:
        raise ValueError("Nie znaleziono tagu <body> w pliku HTML.")

    body_end = lower.find(">", body_idx)
    if body_end == -1:
        raise ValueError("Tag <body> wygląda na uszkodzony (brak '>').")

    new_html = html[:body_end + 1] + "\n" + injection_html + "\n" + html[body_end + 1:]
    Path(output_path).write_text(new_html, encoding="utf-8")

inject_panel_into_html(PLOT_HTML_PATH, OUTPUT_HTML_PATH, INSTRUKCJA_HTML)
print(f"Zapisano: {OUTPUT_HTML_PATH}")


In [ ]:
df_dobowe_zuzycie_ciepla["E"] = df_dobowe_zuzycie_ciepla["Energia T1, GJ"] 

In [ ]:
df_dobowe_zuzycie_ciepla[["Data", "E"]]

In [ ]:
df_dobowe_zuzycie_ciepla["Ep"] = df_dobowe_zuzycie_ciepla["E"].shift(1)

In [ ]:
df_dobowe_zuzycie_ciepla["Eo"] = df_dobowe_zuzycie_ciepla["E"] 

In [ ]:
df_dobowe_zuzycie_ciepla["Ed"] = df_dobowe_zuzycie_ciepla["Eo"] - df_dobowe_zuzycie_ciepla["Ep"]

In [ ]:
df_dobowe_zuzycie_ciepla[["Data", "czas", "Ed", "Eo", "Ep"]]

In [ ]:
import plotly.express as px

fig = px.bar(
    df_dobowe_zuzycie_ciepla,
    x="Data",
    y="Ed",
    title="Dobowe zużycie ciepła [GJ]",
    labels={
        "Data": "Data",
        "Ed": "Dobowe zużycie ciepła Ed [GJ]"
    }
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(tickangle=-45)

fig.write_html("dobowe_zuzycie_ciepla.html", include_plotlyjs="cdn")

In [ ]:
df_dobowe_zuzycie_ciepla[["Data", "czas", "Ed", "Eo", "Ep"]]

In [ ]:
df_dobowe_zuzycie_ciepla["Day"] = pd.to_datetime(df_dobowe_zuzycie_ciepla["Data"]).dt.day
df_czwartego_dnia_miesiaca = df_dobowe_zuzycie_ciepla[df_dobowe_zuzycie_ciepla["Day"] == 4]
df_czwartego_dnia_miesiaca[["Data", "czas", "Ed", "Eo", "Ep"]]

In [ ]:
df_belimo_czas_dt = pd.to_datetime(df_belimo["czas"])
df_belimo["Day"] = df_belimo_czas_dt.dt.day
df_belimo["Hour"] = df_belimo_czas_dt.dt.hour
df_belimo["Date"] = df_belimo_czas_dt.dt.date

# Filtruj dla 5go dnia miesiaca o polnocy (pierwszy wpis kazdego 5go dnia)
df_fifth_day_midnight = df_belimo[(df_belimo["Day"] == 5) & (df_belimo["Hour"] == 0)].drop_duplicates(subset=["Date"], keep="first").copy()

# Sprawdz czy istnieje wpis dla 2025-07-05
july_5_exists = any(df_fifth_day_midnight["Date"] == pd.to_datetime("2025-07-05").date())

# Jesli nie istnieje 2025-07-05 o polnocy, dodaj ostatni wpis z 2025-07-04 (12:01)
if not july_5_exists:
    last_july_4 = df_belimo[df_belimo["Date"] == pd.to_datetime("2025-07-04").date()]
    if not last_july_4.empty:
        # Weź ostatni wiersz z 2025-07-04
        last_entry = last_july_4.iloc[-1:].copy()
        last_entry["Date"] = pd.to_datetime("2025-07-05").date()
        df_fifth_day_midnight = pd.concat([df_fifth_day_midnight, last_entry], ignore_index=True)


In [ ]:

df_fifth_day_midnight["Ek"] = df_fifth_day_midnight["Energia T1, GJ"]
df_fifth_day_midnight["En"] = df_fifth_day_midnight["Ek"].shift(1)

In [ ]:
df_fifth_day_midnight["Em"] = df_fifth_day_midnight["Ek"] - df_fifth_day_midnight["En"] 

In [ ]:
df_fifth_day_midnight[["Data", "Em"]]

In [ ]:
def format_zakres_column(df):
    df["Zakres"] = (
        df["start"].dt.strftime("%d.%m")
        + " - " +
        df["end"].dt.strftime("%d.%m")
    )

def add_zakres_column(df):
    df["start"] = pd.to_datetime(df["Data"]) - pd.DateOffset(months=1)
    df["end"] = pd.to_datetime(df["Data"]) - pd.DateOffset(days=1)

    format_zakres_column(df)

def get_start(x):
    if x == "Kwiecień":
        return pd.to_datetime("2025-04-05")
    elif x == "Maj":
        return pd.to_datetime("2025-05-05")
    elif x == "Czerwiec":
        return pd.to_datetime("2025-06-05")
    else:
        return pd.NaT

def get_end(x):
    if x == "Kwiecień":
        return pd.to_datetime("2025-05-04")
    elif x == "Maj":
        return pd.to_datetime("2025-06-04")
    elif x == "Czerwiec":
        return pd.to_datetime("2025-07-04")
    else:
        return pd.NaT
    

def add_zakres_column_hardcoded(df):
    df["start"] = df["miesiac"].map(get_start)
    df["end"] = df["miesiac"].map(get_end)

    format_zakres_column(df)

In [ ]:
import plotly.express as px

# Dodaj kolumnę z nazwą poprzedniego miesiąca
df_fifth_day_midnight["Miesiąc"] = (pd.to_datetime(df_fifth_day_midnight["Data"]) - pd.DateOffset(months=1)).dt.strftime("%B %Y")

df_fifth_day_midnight["start"] = pd.to_datetime(df_fifth_day_midnight["Data"]) - pd.DateOffset(months=1)
df_fifth_day_midnight["end"] = pd.to_datetime(df_fifth_day_midnight["Data"]) - pd.DateOffset(days=1)

df_fifth_day_midnight["Zakres"] = (
    df_fifth_day_midnight["start"].dt.strftime("%d.%m")
    + " - " +
    df_fifth_day_midnight["end"].dt.strftime("%d.%m")
)

# Usuń pierwszy wiersz (kwiecień - nie ma poprzedniego miesiąca)
df_miesieczne = df_fifth_day_midnight.iloc[1:].copy()

fig = px.bar(
    df_miesieczne,
    x="Zakres",
    y="Em",
    title="Miesięczne zużycie ciepła [GJ]",
    labels={
        "Zakres": "2025",
        "Em": "Miesięczne zużycie ciepła Em [GJ]"
    }
)
fig.update_traces(width=0.2)
fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(tickangle=-45)

fig.write_html("miesieczne_zuzycie_ciepla.html", include_plotlyjs="cdn")

In [ ]:
# Oblicz różnicę między maksymalną a minimalną wartością energii w całym df_belimo
Es = df_belimo["Energia T1, GJ"].max() - df_belimo["Energia T1, GJ"].min()
Es

In [ ]:
import plotly.express as px

# Wykres jednowartościowy dla Es
df_es = pd.DataFrame({"Sezon": ["Sezon"], "Es": [Es]})

fig = px.bar(
    df_es,
    x="Sezon",
    y="Es",
    title="Zużycie ciepła sezonowe [GJ]",
    labels={"Sezon": "Sezon kwiecień-lipiec 2025", "Es": "Sezonowe zużycie ciepła Es [GJ]"}
)

fig.update_layout(
    height=400,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_traces(width=0.13)
#fig.update_xaxes(title_text="Sezon")

fig.write_html("sezonowe_zuzycie_ciepla.html", include_plotlyjs="cdn")

In [ ]:
df_belimo["T1"] = df_belimo["T1, st.C"]

In [ ]:
df_belimo["T2"] = df_belimo["T2, st.C"]

In [ ]:
df_belimo["Tsr"] = (df_belimo["T1"] + df_belimo["T2"]) / 2

In [ ]:
filtered_df = df_belimo[df_belimo["Przepływ, m3/h"] > 0]

filtered_df['czas'] = pd.to_datetime(filtered_df['czas'])

hourly_df = filtered_df.set_index('czas').resample('H')['Tsr'].mean().reset_index()

In [ ]:
import plotly.express as px



fig = px.bar(
    hourly_df,
    x="czas",
    y="Tsr",
    title="Temperatura czynnika na zasilaniu pompy ciepła (średnia godzinowa) podczas pracy pompy ciepła",
    labels={"czas": "Czas", "Tsr": "Tśr [°C]"},
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_traces(width=3600000, marker_color="darkblue")

fig.update_xaxes(rangeslider_visible=True, tickangle=-45)

fig.write_html("temperatura_czynnika_po_stronie_sieciowej.html", include_plotlyjs="cdn")

In [ ]:
K = 273.15

In [ ]:
filtered_df["ro_w"] = 1000 - (0.00198582 * (filtered_df["Tsr"] + K + 10) * (filtered_df["Tsr"] + K - 277)**2) / (filtered_df["Tsr"] + K - 205.8)

In [ ]:
filtered_df[["ro_w", "Tsr"]]

In [ ]:
filtered_df["Tsr_rounded"] = filtered_df["Tsr"].round(2)

In [ ]:
filtered_df["Tsr_rounded"]

In [ ]:
filtered_df.describe()

In [ ]:
df_belimo.describe()

In [ ]:
import pandas as pd

df_interpolacja = pd.read_csv("data/interpolacja_wody.csv")

df_interpolacja

In [ ]:
# Lookup cw from df_interpolacja based on Tsr_rounded
cw_series = df_interpolacja.set_index('Temperatura (°C)')['Cieplo wlasciwe cp (kJ/(kg·K))']
filtered_df['cw'] = filtered_df['Tsr_rounded'].map(cw_series)

filtered_df[['Tsr_rounded', 'cw']].head()

In [ ]:
filtered_df["Q_cwu_sr"] = (filtered_df["Przepływ, m3/h"] * filtered_df["ro_w"] * (filtered_df["T1, st.C"] - filtered_df["T2, st.C"]) * filtered_df["cw"]) / 3600

In [ ]:
filtered_df[filtered_df["T1, st.C"] >  filtered_df["T2, st.C"]][["Q_cwu_sr", "Tsr", "T1, st.C", "T2, st.C", "Przepływ, m3/h"]]

In [ ]:
filtered_df[filtered_df["T1, st.C"] >  filtered_df["T2, st.C"]]

In [ ]:
import numpy as np

# Create filtered_df_with_nans with all rows from df_belimo, adding NaNs for calculated columns where flow <= 0
filtered_df_with_nans = df_belimo.copy()

# Initialize calculated columns with NaN
filtered_df_with_nans['Tsr'] = np.nan
filtered_df_with_nans['ro_w'] = np.nan
filtered_df_with_nans['Tsr_rounded'] = np.nan
filtered_df_with_nans['cw'] = np.nan
filtered_df_with_nans['Q_cwu_sr'] = np.nan

# Update where flow > 0
mask = filtered_df_with_nans['Przepływ, m3/h'] > 0
filtered_df_with_nans.loc[mask, ['Tsr', 'ro_w', 'Tsr_rounded', 'cw', 'Q_cwu_sr']] = filtered_df[['Tsr', 'ro_w', 'Tsr_rounded', 'cw', 'Q_cwu_sr']].values

filtered_df_with_nans = filtered_df_with_nans[filtered_df_with_nans["T1, st.C"] > filtered_df_with_nans["T2, st.C"]]

In [ ]:
filtered_df[["czas", "Tsr", "T1", "T2", "Przepływ, m3/h", "Q_cwu_sr", "ro_w", "cw"]][filtered_df["T1"] > filtered_df["T2"]]

In [601]:
filtered_df_with_nans_to_display = filtered_df_with_nans.fillna(0)

C:\Users\Martyna\AppData\Local\Temp\ipykernel_10084\1287266285.py:1: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [608]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])





# 🔵 Linia 1 — moc cieplna
fig.add_trace(
    go.Scatter(
        x=filtered_df_with_nans_to_display["czas"],
        y=filtered_df_with_nans_to_display["Q_cwu_sr"],
        name="Q_cwu_śr [kW]",
        mode="lines"
    ),
    secondary_y=False,
)

# 🔴 Linia 2 — przepływ (inna jednostka)
fig.add_trace(
    go.Scatter(
        x=filtered_df_with_nans_to_display["czas"],
        y=filtered_df_with_nans_to_display["Przepływ, m3/h"], 
        name="Przepływ [m³/h]",
        mode="lines",
        hovertemplate="%{y}"
    ),
    secondary_y=True,
)

# 🔧 Layout
fig.update_layout(
    title="Zależność mocy na wymienniku cwu od przepływu na przewodzie zasilającym pompę ciepła",
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=70, t=60, b=50),
)

# 🔧 Osie
fig.update_xaxes(rangeslider_visible=True, tickangle=-45)

fig.update_yaxes(title_text="Q_cwu_śr [kW]", secondary_y=False)
fig.update_yaxes(title_text="Przepływ [m³/h]", secondary_y=True)

# 💾 zapis
fig.write_html("q_cwu_sr_full.html", include_plotlyjs="cdn")

In [ ]:
df_kamstrup7

In [ ]:
df_kamstrup7["T1"] = df_kamstrup7["Unnamed: 18"]

df_kamstrup7["T2"] = df_kamstrup7["Unnamed: 19"]

In [ ]:
df_kamstrup7[["czas", "T1", "T2"]]

In [ ]:
df_kamstrup7[["czas", "T1", "T2"]][df_kamstrup7["T1"] > df_kamstrup7["T2"] + 0.1]

In [ ]:
df_kamstrup7[["czas", "T1", "T2"]][df_kamstrup7["T1"] + 0.1 < df_kamstrup7["T2"]]

In [ ]:
df_kamstrup7["V"] = df_kamstrup7["Unnamed: 8"]

In [ ]:
df_kamstrup7["V"]

In [ ]:
df_kamstrup7["V_prev"] = df_kamstrup7["V"].shift(1)

In [ ]:
df_kamstrup7["dV"] = df_kamstrup7["V"] - df_kamstrup7["V_prev"]

In [ ]:
df_kamstrup7_flow_only = df_kamstrup7[df_kamstrup7["dV"] > 0]

In [ ]:
df_kamstrup7_flow_only_filtered = df_kamstrup7_flow_only[df_kamstrup7_flow_only["T1"] > df_kamstrup7_flow_only["T2"] + 0.1]

In [ ]:
df_kamstrup7_flow_only_filtered

In [ ]:
df_kamstrup7_flow_only[[ "dV", "T1", "T2"]][df_kamstrup7_flow_only["T1"] > df_kamstrup7_flow_only["T2"] ].mean()

In [ ]:
filtered_df = filtered_df[filtered_df["Przepływ, m3/h"] > 1]

In [ ]:
filtered_df_with_nans['czas'] = pd.to_datetime(filtered_df_with_nans['czas'])
df_filtered_belimo_hourly = filtered_df.set_index('czas').resample('h').mean(numeric_only=True).reset_index()

In [ ]:
df_filtered_belimo_hourly

In [ ]:
df_kamstrup7["T1"] = df_kamstrup7["Unnamed: 18"]
df_kamstrup7["T2"] = df_kamstrup7["Unnamed: 19"]

In [ ]:
df_kamstrup7_hourly = df_kamstrup7.set_index('czas').resample('h')[['T1', 'T2']].mean().reset_index()
df_filtered_belimo_hourly = df_filtered_belimo_hourly.merge(df_kamstrup7_hourly, on='czas', how='left', suffixes=('', '_kamstrup'))

In [ ]:
df_filtered_belimo_hourly["mi_raw"] =100 * (df_filtered_belimo_hourly["T1_kamstrup"] - df_filtered_belimo_hourly["T2_kamstrup"])/(df_filtered_belimo_hourly["T1"] - df_filtered_belimo_hourly["T2"])

In [ ]:
df_filtered_belimo_hourly["mi"] = np.where((df_filtered_belimo_hourly["Przepływ, m3/h"] < 1) | (df_filtered_belimo_hourly["mi_raw"] < 0), np.nan, df_filtered_belimo_hourly["mi_raw"])

In [ ]:
df_filtered_belimo_hourly[["czas", "Przepływ, m3/h", "mi", "mi_raw", "T1", "T2", "T1_kamstrup", "T2_kamstrup"]]

In [ ]:
df_filtered_belimo_hourly[["czas", "Przepływ, m3/h", "mi", "mi_raw", "T1", "T2", "T1_kamstrup", "T2_kamstrup"]]

# Analyze 'mi' column
nan_count = df_filtered_belimo_hourly['mi'].isna().sum()
between_0_100 = ((df_filtered_belimo_hourly['mi'] >= 0) & (df_filtered_belimo_hourly['mi'] <= 100)).sum()
less_than_0 = (df_filtered_belimo_hourly['mi'] < 0).sum()
more_than_100 = (df_filtered_belimo_hourly['mi'] > 100).sum()

print(f"NaN values in 'mi': {nan_count}")
print(f"Values between 0 and 100: {between_0_100}")
print(f"Values less than 0: {less_than_0}")
print(f"Values more than 100: {more_than_100}")
print(f"Total rows: {len(df_filtered_belimo_hourly)}")

In [ ]:
df_filtered_belimo_hourly[["czas", "Przepływ, m3/h", "mi", "mi_raw", "T1", "T2", "T1_kamstrup", "T2_kamstrup"]]

In [ ]:
import plotly.express as px

fig = px.bar(
    df_filtered_belimo_hourly,
    x="czas",
    y="mi",
    title="Sprawność wymiennika CWU",
    labels={"czas": "Czas", "mi": "Sprawność [%]"}
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(rangeslider_visible=True, tickangle=-45)

fig.update_traces(width=3600000, marker_color="darkblue")

fig.write_html("sprawnosc_wymiennika_cwu.html", include_plotlyjs="cdn")

In [ ]:
df_belimo_for_mi = df_belimo.copy()

In [ ]:
df_belimo_for_mi = df_belimo.copy()
df_belimo_for_mi['czas'] = pd.to_datetime(df_belimo_for_mi['czas'])
df_belimo_for_mi['czas_hour'] = df_belimo_for_mi['czas'].dt.floor('H')

In [ ]:
df_belimo_for_mi["T1"] = df_belimo_for_mi["T1, st.C"]
df_belimo_for_mi["T2"] = df_belimo_for_mi["T2, st.C"]

In [ ]:
df_kamstrup7['czas'] = pd.to_datetime(df_kamstrup7['czas'])
df_kamstrup7['czas_hour'] = df_kamstrup7['czas'].dt.floor('H')

df_belimo_for_mi = df_belimo_for_mi.merge(df_kamstrup7[['czas_hour', 'T1', 'T2']], on='czas_hour', how='left', suffixes=('', '_kamstrup'))

In [ ]:
df_belimo_for_mi[["czas", "Przepływ, m3/h", "T1", "T2", "T1_kamstrup", "T2_kamstrup"]][df_belimo_for_mi["Przepływ, m3/h"] > 1]

In [ ]:

# Summarize mi_raw column
nan_count = df_belimo_for_mi_filtered['mi_raw'].isna().sum()
less_than_0 = (df_belimo_for_mi_filtered['mi_raw'] < 0).sum()
between_0_100 = ((df_belimo_for_mi_filtered['mi_raw'] >= 0) & (df_belimo_for_mi_filtered['mi_raw'] <= 100)).sum()
more_than_100 = (df_belimo_for_mi_filtered['mi_raw'] > 100).sum()

print(f"NaN values in 'mi_raw': {nan_count}")
print(f"Values less than 0: {less_than_0}")
print(f"Values between 0 and 100: {between_0_100}")
print(f"Values more than 100: {more_than_100}")
print(f"Total rows: {len(df_belimo_for_mi_filtered)}")

In [ ]:
df_belimo_for_mi_filtered = df_belimo_for_mi[df_belimo_for_mi["Przepływ, m3/h"] > 1]

df_belimo_for_mi_filtered['mi'] = np.where(
    (df_belimo_for_mi_filtered['mi_raw'] >= 0) & (df_belimo_for_mi_filtered['mi_raw'] <= 100),
    df_belimo_for_mi_filtered['mi_raw'],
    np.nan
)

In [ ]:
df_belimo_for_mi_filtered = (
    df_belimo_for_mi_filtered
    .set_index("czas")
    .reindex(df_belimo["czas"])
    .reset_index()
)

In [ ]:
df_belimo_for_mi_filtered

In [ ]:
df_belimo_for_mi_filtered = df_belimo_for_mi[(df_belimo_for_mi["Przepływ, m3/h"] > 1) & (df_belimo_for_mi_filtered["T1"] != df_belimo_for_mi_filtered["T2"])]

In [ ]:
df_belimo_for_mi_filtered[["czas", "Przepływ, m3/h", "T1", "T2", "T1_kamstrup", "T2_kamstrup"]][df_belimo_for_mi_filtered["T1"] == df_belimo_for_mi_filtered["T2"]]

In [ ]:
df_belimo_for_mi_filtered["mi_raw"] =100 * (df_belimo_for_mi_filtered["T1_kamstrup"] - df_belimo_for_mi_filtered["T2_kamstrup"])/(df_belimo_for_mi_filtered["T1"] - df_belimo_for_mi_filtered["T2"])

In [ ]:
df_belimo_for_mi_filtered[["mi_raw", "czas", "Przepływ, m3/h", "T1", "T2", "T1_kamstrup", "T2_kamstrup"]]

In [ ]:
import plotly.express as px

fig = px.line(
    df_belimo_for_mi_filtered,
    x="czas",
    y="mi",
    title="Sprawność wymiennika cwu",
    labels={"czas": "Czas", "mi": "Sprawność [%]"}
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(rangeslider_visible=True, tickangle=-45)

fig.write_html("sprawnosc_wymiennika_vol2.html", include_plotlyjs="cdn")

In [ ]:
df_belimo["V_niezero"] = df_belimo["Przepływ, m3/h"].map(lambda x : 1 if x > 0 else 0)

In [ ]:
df_belimo[["Data", "V_niezero"]]

In [ ]:
df_belimo_volume_per_day = (
    df_belimo[["Data", "V_niezero"]]
    .groupby("Data", as_index=False)["V_niezero"]
    .sum()
)

df_belimo_volume_per_day_count = (
    df_belimo[["Data", "V_niezero"]]
    .groupby("Data", as_index=False).count()
)

df_belimo_volume_per_day["count"] = df_belimo_volume_per_day_count["V_niezero"]

In [ ]:
df_belimo_volume_per_day["percent_volume"] = 100 * (df_belimo_volume_per_day["V_niezero"] / df_belimo_volume_per_day["count"])

In [ ]:
df_belimo_volume_per_day

In [ ]:
import plotly.express as px

fig = px.bar(
    df_belimo_volume_per_day,
    x="Data",
    y="percent_volume",
    title="Dobowy czas pracy pompy ciepła",
    labels={"Data": "Data", "percent_volume": "Czas pracy [%]"}
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(rangeslider_visible=True, tickangle=-45)

fig.write_html("ilosc_minut_pracy_pompy_ciepla_dziennie_procenty.html", include_plotlyjs="cdn")

In [ ]:
df_belimo_volume_per_day

In [ ]:
import plotly.express as px

fig = px.bar(
    df_belimo_volume_per_day,
    x="Data",
    y="V_niezero",
    title="Ilość minut pracy pompy ciepła dziennie",
    labels={"Data": "Data", "V_niezero": "Ilość minut"}
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(rangeslider_visible=True, tickangle=-45)

fig.write_html("ilosc_minut_pracy_pompy_ciepla_dziennie.html", include_plotlyjs="cdn")

In [ ]:
df_belimo_volume_per_month = (
    df_belimo_volume_per_day
    .assign(miesiac_rozliczeniowy=(pd.to_datetime(df_belimo_volume_per_day["Data"]) - pd.Timedelta(days=4)).dt.to_period("M"))
    .groupby("miesiac_rozliczeniowy", as_index=False)
    .sum(numeric_only=True)
)

In [ ]:
df_belimo_volume_per_month["percent_volume"] = (df_belimo_volume_per_month["V_niezero"] / df_belimo_volume_per_month["count"]) * 100

In [ ]:
df_belimo_volume_per_month["miesiac"] = (
    pd.to_datetime(df_belimo_volume_per_month["miesiac_rozliczeniowy"].astype(str))
    .dt.month_name(locale="pl_PL")
)

In [ ]:
df_belimo_volume_per_month

In [ ]:
import plotly.express as px


fig = px.bar(
    df_belimo_volume_per_month,
    x="Zakres",
    y="V_niezero",
    title="Czas pracy pompy ciepła w miesiącu",
    labels={
        "Zakres": "2025",
        "V_niezero": "Czas pracy pompy ciepła [min]"
    }
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)


fig.update_xaxes(tickangle=-45)

fig.update_traces(width=0.2)

fig.write_html("miesieczna_ilosc_minut_pompy_pracy.html", include_plotlyjs="cdn")

In [ ]:
add_zakres_column_hardcoded(df_belimo_volume_per_month)

In [ ]:
df_belimo_volume_per_month

In [ ]:
# policz sumy
sum_row = pd.DataFrame([{
    "miesiac_rozliczeniowy": "Sezon",
    "V_niezero": df_belimo_volume_per_month["V_niezero"].sum(),
    "count": df_belimo_volume_per_month["count"].sum(),
    "percent_volume": df_belimo_volume_per_month["V_niezero"].sum() / df_belimo_volume_per_month["count"].sum() * 100,
    "miesiac": "Sezon",
    "start": df_belimo_volume_per_month["start"].min(),
    "end": df_belimo_volume_per_month["end"].max(),
    "Zakres": "Cały sezon"
}])

# dodaj do df_belimo_volume_per_month
df_belimo_volume_per_month_with_season = pd.concat([df_belimo_volume_per_month, sum_row], ignore_index=True)

In [ ]:
df_belimo_volume_per_month_with_season

In [ ]:
import plotly.express as px


fig = px.bar(
    df_belimo_volume_per_month_with_season,
    x="Zakres",
    y="percent_volume",
    title="Czas pracy pompy ciepła w miesiącu",
    labels={"Zakres": "2025", "percent_volume": "Czas pracy [%]"}
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    margin=dict(l=70, r=20, t=60, b=50),
)

fig.update_xaxes(tickangle=-45)

fig.update_traces(width=0.2)

fig.write_html("miesieczna_ilosc_minut_pompy_pracy_procenty.html", include_plotlyjs="cdn")


In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu = (
    df_kamstrup7[["czas", "T1"]]
    .merge(
        df[["czas", "T2"]],
        on="czas",
        how="outer"  # zachowuje wszystkie punkty z obu df
    )
    .sort_values("czas")
)

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu = df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu.sort_values("czas").ffill()

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu = df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu.rename(columns={
    "T2": "APEK - Temperatura T2 [°C]",
    "T1": "Kamstrup 207 Temperatura T1 [°C]"
})

fig = px.line(
    df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu,
    x="czas",
    y=["APEK - Temperatura T2 [°C]", "Kamstrup 207 Temperatura T1 [°C]"],
    title="Zależność temperatury ciepłej wody od temperatury na wyjściu z pompy ciepła",
    labels={"value": "Temperatura", "czas": "Czas", "variable": "Seria"}
)

fig.update_layout(height=500, hovermode="x unified")

fig.update_xaxes(rangeslider_visible=True)

fig.write_html("zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu.html", include_plotlyjs="cdn")

In [ ]:
df_belimo["czas"] = pd.to_datetime(df_belimo["czas"])
df_kamstrup7["czas"] = pd.to_datetime(df_kamstrup7["czas"])
df_pec["czas"] = pd.to_datetime(df_pec["czas"])

df_pec["T2"] = df_pec["T2"].astype(float)

df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2 = (
    df_belimo[["czas", "T1"]]
    .merge(
        df_kamstrup7[["czas", "T1"]],
        on="czas",
        how="outer" ,
         
        suffixes=("_belimo", "_kamstrup") # zachowuje wszystkie punkty z obu df
    ).merge(
        df_pec[["czas", "T2"]],
        on="czas",
        how="outer" ,
    )
    .sort_values("czas")
)

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2 = df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2.sort_values("czas").ffill()

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2_to_plot

In [ ]:
print(df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2_to_plot[
    ["Belimo - Temperatura T1 [°C]", "Kamstrup 207 - Temperatura T1 [°C]", "PEC - Temperatura T2 [°C]"]
].dtypes)

In [ ]:
df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2_to_plot = df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2.rename(columns={
    "T1_kamstrup": "Kamstrup 207 - Temperatura T1 [°C]",
    "T1_belimo": "Belimo - Temperatura T1 [°C]",
    "T2": "PEC - Temperatura T2 [°C]"
})

fig = px.line(
    df_zaleznosc_temperatury_cieplnej_wody_od_temp_na_wyjsciu_2_to_plot,
    x="czas",
    y=["Belimo - Temperatura T1 [°C]", "Kamstrup 207 - Temperatura T1 [°C]", "PEC - Temperatura T2 [°C]"],
    title=" Zależność temperatury na zasilaniu podgrzewacza cwu od temperatury zasilania pompy ciepła i temperatury na zasilaniu sieci ciepłowniczej",
    labels={"value": "Temperatura", "czas": "Czas", "variable": "Seria"}
)

fig.update_layout(height=500, hovermode="x unified")

fig.update_xaxes(rangeslider_visible=True)

fig.write_html("tcwu_tczas.html", include_plotlyjs="cdn")